# IndexTTS2 - 部署笔记本 (Colab/Kaggle)

**解决导入悖论**: 必须先克隆代码，才能使用deploy模块

**mamba优先**: 先安装mamba，再用mamba快速安装依赖

In [ ]:
# ============================================
# Cell 1: 安装mamba + 环境准备 (独立运行)
# ============================================
import os, sys, json, subprocess, urllib.request

# 环境检测
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/input') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
WORK_DIR = '/content' if IN_COLAB else '/kaggle/working' if IN_KAGGLE else '/tmp'
REPO_DIR = f"{WORK_DIR}/index-tts"
MAMBA_ROOT = f"{WORK_DIR}/mamba"

print(f"🔍 环境: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Other'}")
print(f"📁 工作目录: {WORK_DIR}")

# ===== 工具函数 =====
def run_cmd(cmd, check=True):
    """运行命令"""
    result = subprocess.run(cmd, capture_output=True, text=True, shell=isinstance(cmd, str))
    if check and result.returncode != 0:
        print(f"命令失败: {cmd}")
        print(f"错误: {result.stderr}")
        return False
    return True

def check_mamba():
    """检查mamba是否可用且是包管理器"""
    try:
        result = subprocess.run(["mamba", "--version"], capture_output=True, text=True)
        if result.returncode != 0:
            return False
        # 排除测试框架
        output = result.stdout.lower() + result.stderr.lower()
        return 'mamba' in output and 'coverage' not in output and 'test' not in output
    except:
        return False

def install_mamba_micromamba():
    """通过micromamba安装mamba (不依赖conda)"""
    print("🔧 安装 micromamba (独立mamba)...")
    
    # 下载 micromamba 安装脚本
    installer_url = "https://micro.mamba.pm/install.sh"
    install_script = f"{WORK_DIR}/micromamba_install.sh"
    
    try:
        urllib.request.urlretrieve(installer_url, install_script)
        os.chmod(install_script, 0o755)
        
        # 设置安装路径并执行
        env = os.environ.copy()
        env['MAMBA_ROOT_PREFIX'] = MAMBA_ROOT
        
        result = subprocess.run(['bash', install_script], env=env, capture_output=True, text=True)
        if result.returncode == 0:
            print("✅ micromamba 安装完成")
            # 添加micromamba到路径 (micromamba就是mamba的轻量版)
            os.environ['PATH'] = f"{MAMBA_ROOT}/bin:{os.environ['PATH']}"
            # 创建mamba别名
            mamba_bin = f"{MAMBA_ROOT}/bin/micromamba"
            if os.path.exists(mamba_bin):
                os.symlink(mamba_bin, f"{MAMBA_ROOT}/bin/mamba")
                os.environ['PATH'] = f"{MAMBA_ROOT}/bin:{os.environ['PATH']}"
            return True
        else:
            print(f"⚠️ micromamba安装失败: {result.stderr}")
            return False
    except Exception as e:
        print(f"⚠️ micromamba安装异常: {e}")
        return False

def install_mamba_conda():
    """通过conda安装mamba (如果conda存在)"""
    print("🔧 尝试通过conda安装mamba...")
    try:
        result = subprocess.run(["conda", "--version"], capture_output=True, text=True)
        if result.returncode != 0:
            print("⚠️ conda不可用")
            return False
        
        # 使用conda安装mamba
        subprocess.run(["conda", "install", "-y", "-c", "conda-forge", "mamba"], check=True)
        print("✅ mamba (通过conda) 安装完成")
        return True
    except Exception as e:
        print(f"⚠️ 通过conda安装mamba失败: {e}")
        return False

def install_mamba():
    """安装mamba (优先micromamba，回退conda)"""
    if check_mamba():
        print("✅ mamba已可用")
        return True
    
    # 方法1: 安装micromamba (独立，不依赖conda)
    if install_mamba_micromamba():
        return True
    
    # 方法2: 通过conda安装
    if install_mamba_conda():
        return True
    
    print("⚠️ 无法安装mamba，将使用pip")
    return False

def install_with_mamba(packages, channels=None):
    """使用mamba安装包"""
    if not check_mamba():
        return False
    
    print(f"🚀 使用mamba安装: {packages}")
    try:
        cmd = ["mamba", "install", "-y"]
        if channels:
            for ch in channels:
                cmd.extend(["-c", ch])
        cmd.extend(packages)
        subprocess.run(cmd, check=True)
        return True
    except Exception as e:
        print(f"⚠️ mamba安装失败: {e}")
        return False
# ===================================================

# 步骤1: 安装mamba
print("\n📦 步骤1: 安装mamba包管理器...")
MAMBA_AVAILABLE = install_mamba()

# 步骤2: 使用mamba安装python和cuda (比pip快很多)
print("\n📦 步骤2: 安装基础环境...")
if MAMBA_AVAILABLE:
    # 使用mamba快速安装
    deps = ["python=3.10", "cudatoolkit=11.8"]
    channels = ["conda-forge", "nvidia"]
    success = install_with_mamba(deps, channels)
    if not success:
        print("⚠️ mamba安装失败，跳过")
else:
    print("ℹ️ mamba不可用，跳过此步骤")

# 步骤3: 安装PyTorch (必须通过pip)
print("\n📦 步骤3: 安装PyTorch...")
subprocess.run(["pip", "install", "-q", "torch==2.8.0", "torchaudio==2.8.0", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
print("✅ PyTorch安装完成")

# 步骤4: 安装其他基础包
print("\n📦 步骤4: 安装基础依赖...")
basic_deps = ["pyngrok", "gradio==5.45.0", "fastapi", "uvicorn", "python-multipart", "huggingface-hub"]
for dep in basic_deps:
    subprocess.run(["pip", "install", "-q", dep], check=True)
print("✅ 基础依赖安装完成")

# 步骤5: 克隆代码
print("\n📥 步骤5: 克隆代码...")
if os.path.exists(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "-b", "dev", "https://github.com/infinite-gaming-studio/index-tts.git", REPO_DIR], check=True)
print(f"✅ 代码克隆完成")

# 保存配置
config = {
    "work_dir": WORK_DIR,
    "repo_dir": REPO_DIR, 
    "in_colab": IN_COLAB,
    "in_kaggle": IN_KAGGLE,
    "mamba_available": MAMBA_AVAILABLE
}
json.dump(config, open("/tmp/notebook_config.json", "w"))

# 添加项目路径
sys.path.insert(0, REPO_DIR)

print(f"\n{'='*50}")
print(f"✅ 环境准备完成!")
print(f"   mamba: {'✅ 可用' if MAMBA_AVAILABLE else '❌ 不可用'}")
print(f"   代码路径: {REPO_DIR}")
print(f"\n👉 现在可以运行Cell 2")
print(f"{'='*50}")

In [ ]:
# ============================================
# Cell 2: 完整安装 + 模型下载
# ============================================

# 加载配置
config = json.load(open("/tmp/notebook_config.json"))
IN_COLAB = config.get("in_colab", False)
IN_KAGGLE = config.get("in_kaggle", False)
MAMBA_AVAILABLE = config.get("mamba_available", False)

# 导入deploy模块
from deploy.utils import DependencyInstaller, ModelDownloader, check_model_exists

# 安装项目依赖
print("📦 安装项目依赖...")
DependencyInstaller.install_deps(
    use_mamba=MAMBA_AVAILABLE,  # 如果Cell 1安装了mamba就使用
    in_colab=IN_COLAB,
    in_kaggle=IN_KAGGLE
)

# 安装项目本身
print("📦 安装项目...")
subprocess.run(["pip", "install", "-q", "-e", REPO_DIR], check=True)

# 下载模型
CHECKPOINT_DIR = f"{REPO_DIR}/checkpoints"
if not check_model_exists(CHECKPOINT_DIR):
    print("\n🤖 下载模型 (约3-8GB)...")
    ModelDownloader.download(CHECKPOINT_DIR, source="huggingface")
else:
    print("✅ 模型已存在")

print(f"\n{'='*50}")
print("✅ 全部准备完成! 可以启动服务了")
print(f"{'='*50}")

In [ ]:
# ============================================
# Cell 3: 启动服务
# ============================================
from deploy.launcher import quick_start

PORT = 8000
MODE = "both"  # "api" | "webui" | "both"
NGROK_TOKEN = None  # 替换为你的token获取公网访问

launcher = quick_start(port=PORT, mode=MODE, ngrok_token=NGROK_TOKEN)

In [ ]:
# ============================================
# Cell 4: 服务管理 (可选)
# ============================================

launcher.logs(lines=30)

# launcher.status()
# launcher.stop()